# Train + evaluate DistilBERT (free Colab GPU)
**Runtime → Change runtime type → T4 GPU** (free). Do not pick a paid GPU, and ignore any Colab Pro / compute-unit offers.

Before running: upload `train.csv`, `val.csv`, `test.csv` from your Mac's `data/processed/` to
`MyDrive/airbnb_nlp/data/processed/` in Google Drive. For cell 8, add a Hugging Face **Write** token as the
Colab secret `HF_TOKEN` (key icon in the left sidebar).

Run the cells one at a time, in order. Checkpoints go to Drive every epoch; if Colab disconnects,
run cells 2, 3 and 5 again and training resumes.


In [ ]:
# 1. Check the GPU and Python version
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

In [ ]:
# 2. Mount Google Drive (free) and set paths
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/airbnb_nlp'
DATA = f'{DRIVE}/data/processed'
MODELS = f'{DRIVE}/models/distilbert'
REPORTS = f'{DRIVE}/reports'
import os
print(sorted(os.listdir(DATA)))   # should show test.csv, train.csv, val.csv

In [ ]:
%cd /content
# 3. Get the code from GitHub (same commit as your Mac) and install it
!rm -rf /content/repo
!git clone -q -b nlp-finetuning https://github.com/sgaind-data/Airbnb_Review_Scores_ML.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -e ".[train]"

In [ ]:
# 4. Smoke test: 1 epoch on 500 rows (about 1 minute). Catches errors before the real run.
!python -m airbnb_nlp.train --data-dir {DATA} --output-dir /content/smoke --epochs 1 --limit 500

In [ ]:
# 5. Full training: 3 epochs, checkpoints saved to Drive (about 10-20 minutes on a T4)
!python -m airbnb_nlp.train --data-dir {DATA} --output-dir {MODELS}

In [ ]:
# 6. Test-set evaluation vs VADER, confusion matrix, error analysis
!python -m airbnb_nlp.evaluate --data-dir {DATA} --model-dir {MODELS}/final --reports-dir {REPORTS}
!cp {MODELS}/train_history.json {REPORTS}/

In [ ]:
# 7. Look at the results
from IPython.display import Image, Markdown, display
import pandas as pd
display(Markdown(open(f'{REPORTS}/comparison.md').read()))
display(Image(f'{REPORTS}/confusion_matrix.png'))
pd.set_option('display.max_colwidth', 300)
pd.read_csv(f'{REPORTS}/error_analysis.csv')

In [ ]:
# 8. Publish the model to the Hugging Face Hub (free) with a model card
import json, shutil
from google.colab import userdata
from huggingface_hub import HfApi
from transformers import pipeline

HF_USER = "sgaind-dataai"
REPO_ID = f"{HF_USER}/distilbert-airbnb-listing-quality"
api = HfApi(token=userdata.get("HF_TOKEN"))
print("Logged in as:", api.whoami()["name"])

comparison = open(f"{REPORTS}/comparison.md").read()
card = open("/content/repo/docs/MODEL_CARD.md").read().replace("{{COMPARISON}}", comparison)

stage = "/content/hf_upload"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(f"{MODELS}/final", stage, ignore=shutil.ignore_patterns("training_args.bin"))
open(f"{stage}/README.md", "w").write(card)
api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=stage, repo_id=REPO_ID, commit_message="Upload fine-tuned model and card")
print(f"Published: https://huggingface.co/{REPO_ID}")

clf = pipeline("text-classification", model=REPO_ID)
for t in ["Spotless, gorgeous home and the host was wonderful. Would stay again!",
          "Nice location but the room was dusty and we had to pay extra for parking."]:
    print(clf(t)[0], "<-", t)

In [ ]:
# 9. Download the reports to your Mac (then: unzip -o ~/Downloads/reports.zip -d reports/)
!cd "{REPORTS}" && zip -q -r /content/reports.zip . && unzip -l /content/reports.zip
from google.colab import files
files.download("/content/reports.zip")